# Envoyer le dataset betsileo ↔ malgache sur Hugging Face (depuis Kaggle)

**Avant de lancer**
1. Sur Kaggle : *Datasets → New Dataset*, dépose `train.jsonl`, `validation.jsonl`, `test.jsonl`
   (et `README.md` si tu l'as : c'est la carte du dataset). Donne-lui un nom, par exemple `betsileo-mg-v2`.
2. Dans ce notebook : *Add Input* → ton dataset.
3. *Add-ons → Secrets* : `HF_TOKEN` (token Hugging Face avec droit **write**), activé pour ce notebook.
4. *Settings → Internet* : **On**. Pas besoin de GPU.
5. *Run All*.

In [ ]:
CFG = {
    "REPO_ID": "nirina-doris-sylvie/betsileo-mg-v2",   # depot Hugging Face a creer / mettre a jour
    "PRIVATE": True,                                    # garde le depot prive (droits des textes bibliques)
    "INPUT": "/kaggle/input",                           # ou Kaggle range les datasets attaches
    "PUSH": True,                                       # False = tout verifier sans envoyer
}

## 1. Trouver et vérifier les fichiers

In [ ]:
import glob, json, os, re, shutil
from collections import Counter

SPLITS = ("train", "validation", "test")
FIELDS = {"source", "target", "direction", "ref"}

hits = glob.glob(os.path.join(CFG["INPUT"], "**", "train.jsonl"), recursive=True)
assert hits, "train.jsonl introuvable : as-tu ajoute ton dataset avec Add Input ?"
data_dir = os.path.dirname(hits[0])
print("Fichiers trouves dans", data_dir, ":", sorted(os.listdir(data_dir)))


def origin(ref):
    if ref.startswith("LEX"):
        return "lexique"
    return "chanson" if re.search(r"\.(json|txt):\d+$", ref) else "bible"


rows = {}
for split in SPLITS:
    path = os.path.join(data_dir, f"{split}.jsonl")
    assert os.path.exists(path), f"{split}.jsonl manquant"
    rows[split] = []
    with open(path, encoding="utf-8") as f:
        for n, line in enumerate(f, 1):
            if not line.strip():
                continue
            r = json.loads(line)                       # plante ici si une ligne est cassee
            assert set(r) >= FIELDS, f"{split}.jsonl ligne {n} : champs {sorted(r)}"
            assert r["direction"] in ("Malagasy->Betsileo", "Betsileo->Malagasy"), f"{split}:{n} direction"
            assert r["source"].strip() and r["target"].strip(), f"{split}.jsonl ligne {n} vide"
            rows[split].append(r)
    print(f"{split:<11} {len(rows[split]):>6} lignes  {dict(Counter(origin(r['ref']) for r in rows[split]))}")

# Aucune phrase de validation/test ne doit etre dans le train.
key = lambda s: re.sub(r"\W+", " ", s.lower()).strip()
train_texts = {key(r["target"]) for r in rows["train"]}
for split in ("validation", "test"):
    leaks = [r["ref"] for r in rows[split] if key(r["target"]) in train_texts]
    print(f"fuites train -> {split} : {len(leaks)}", leaks[:5])

## 2. Préparer le dossier à envoyer

In [ ]:
OUT = "/kaggle/working/hf_dataset" if os.path.isdir("/kaggle/working") else os.path.abspath("hf_dataset")
os.makedirs(OUT, exist_ok=True)
for split in SPLITS:
    shutil.copy(os.path.join(data_dir, f"{split}.jsonl"), os.path.join(OUT, f"{split}.jsonl"))

card = os.path.join(data_dir, "README.md")
if os.path.exists(card):
    shutil.copy(card, os.path.join(OUT, "README.md"))
    print("Carte du dataset : README.md fourni")
else:
    n = {s: len(rows[s]) for s in SPLITS}
    with open(os.path.join(OUT, "README.md"), "w", encoding="utf-8") as f:
        f.write(f'''---
language:
- mg
license: other
task_categories:
- translation
pretty_name: Betsileo <-> malgache officiel (v2)
configs:
- config_name: default
  data_files:
  - split: train
    path: train.jsonl
  - split: validation
    path: validation.jsonl
  - split: test
    path: test.jsonl
---

# Betsileo <-> malgache officiel (v2)

Paires paralleles betsileo / malgache officiel, dans les deux sens
(train {n["train"]}, validation {n["validation"]}, test {n["test"]} lignes).

Colonnes : `source` (avec le prefixe `translate Malagasy to Betsileo: ...`), `target`,
`direction`, `ref` (verset `MRK 1:31`, mot du lexique `LEX ...` ou vers de chanson
`Majeur 7/galeoka.json:6`).

Licence : versets issus de traductions bibliques dont les droits sont a verifier
avant toute publication (garder le depot prive).
''')
    print("Carte du dataset : README.md genere")
print(sorted(os.listdir(OUT)))

## 3. Envoyer sur Hugging Face

In [ ]:
def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception as e:   # on affiche la vraie raison donnee par Kaggle
            print("Lecture du secret HF_TOKEN impossible :", type(e).__name__, e)
            token = None
    return (token or "").strip() or None


if CFG["PUSH"]:
    from huggingface_hub import HfApi
    token = get_hf_token()
    assert token, ("Pas de HF_TOKEN. Verifie : 1) Add-ons -> Secrets, nom exact HF_TOKEN, case cochee ; "
                   "2) Internet On ; 3) si le secret a ete ajoute apres le demarrage : "
                   "Run -> Restart & clear cell outputs, puis Run All.")
    api = HfApi(token=token)
    print("Connecte en tant que", api.whoami()["name"])
    api.create_repo(CFG["REPO_ID"], repo_type="dataset", private=CFG["PRIVATE"], exist_ok=True)
    api.upload_folder(repo_id=CFG["REPO_ID"], repo_type="dataset", folder_path=OUT,
                      commit_message="dataset v2 (train / validation / test)")
    print("Envoye : https://huggingface.co/datasets/" + CFG["REPO_ID"])
else:
    print("PUSH = False : rien n'a ete envoye.")

## 4. Vérifier en relisant depuis Hugging Face

In [ ]:
if CFG["PUSH"]:
    from datasets import load_dataset
    ds = load_dataset(CFG["REPO_ID"], token=token)
    print(ds)
    for split in SPLITS:
        assert len(ds[split]) == len(rows[split]), f"{split} : {len(ds[split])} != {len(rows[split])}"
    print("OK : les 3 splits sont en ligne avec le bon nombre de lignes.")
    print(ds["test"][0])